# Notebook 2 — CLS Lens: Individual Patch Token Scoring

**What this notebook does:**
For each patch position `(i, j)` at each target ViT layer `L`:

1. Extract the individual patch token `h[i,j]` ∈ ℝ^d_model
2. Feed it **directly** to the **CLS lens** (trained on CLS tokens): `lens(h[i,j]) → logits`
3. `score[i,j] = softmax(logits)[y_hat]` ∈ [0, 1]

All 16×16 = 256 patches are scored (no border exclusion).

**Note:** the CLS lens was trained on CLS tokens, not patch tokens. This experiment probes
whether raw patch tokens carry class-relevant information that the lens can decode — without
any distribution-shift correction. Compare with Notebook 3 which applies a mean adjustment.

Sanity-check cells (**SC N**) follow every major step.

In [ ]:
%matplotlib inline

import sys
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.datasets as datasets

sys.path.insert(0, str(Path('..').resolve() / 'src'))

from tuned_lens.model import VisionModelWrapper
from tuned_lens.config import ModelConfig
from attribute_lens.scorer import load_lens_checkpoint, discover_lens_files

print('Imports OK')

In [ ]:
# ── CONFIGURATION — edit paths to match your environment ─────────────────────
IMAGENET_VAL_DIR = '/opt/models/datasets/imagenet/extracted/val'

# ── Model selection — uncomment the model you want to use ──────────────────
MODEL_NAME = 'vit_large_patch14_clip_224.openai_ft_in1k'   # CLIP ViT-L/14
# MODEL_NAME = 'vit_large_patch16_224.augreg_in21k_ft_in1k'  # ViT-L/16 (AugReg)
# MODEL_NAME = 'deit3_large_patch16_224.fb_in1k'             # DeiT3-L/16
# MODEL_NAME = 'vit_large_patch14_dinov2.lvd142m'           # DINOv2-L/14    (518×518, 37×37 grid)
_MODEL_SHORT = MODEL_NAME.split('.')[0]  # used to namespace output paths

CLS_LENS_DIR  = f'../outputs/{_MODEL_SHORT}/affine_kld/best_lenses'
TARGET_LAYERS = [0, 6, 12, 18, 23]  # filtered to available checkpoints
N_IMAGES      = 10
SEED          = 42
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'

random.seed(SEED)
torch.manual_seed(SEED)
print(f'Device      : {DEVICE}')
print(f'CLS lens dir: {CLS_LENS_DIR}')

## SC 1 — Verify paths and discover available CLS lens checkpoints

In [ ]:
assert Path(IMAGENET_VAL_DIR).exists(), f'Val dir not found: {IMAGENET_VAL_DIR}'
assert Path(CLS_LENS_DIR).exists(),     f'CLS lens dir not found: {CLS_LENS_DIR}'

available_lenses = discover_lens_files(CLS_LENS_DIR)
print(f'Found {len(available_lenses)} CLS lens checkpoints')
print(f'  Available layers : {sorted(available_lenses.keys())}')

TARGET_LAYERS = sorted(l for l in TARGET_LAYERS if l in available_lenses)
assert TARGET_LAYERS, (
    f'None of the requested layers have checkpoints. '
    f'Found: {sorted(available_lenses.keys())}'
)
print(f'  Using layers     : {TARGET_LAYERS}')
print('SC 1 ✓')

## Load model (patch_mode=True — we need all patch tokens as input)

In [ ]:
model_cfg = ModelConfig(
    model_name=MODEL_NAME,
    pretrained=True,
    target_layers=TARGET_LAYERS,
    freeze_model=True,
    patch_mode=True,   # capture all patch tokens (positions 1:) from each block
)
wrapper = VisionModelWrapper(model_cfg, device=DEVICE)

H, W = wrapper.patch_grid_size
print(f'Model      : {MODEL_NAME}')
print(f'd_model    : {wrapper.d_model}')
print(f'num_classes: {wrapper.num_classes}')
print(f'num_layers : {wrapper.num_layers}')
print(f'patch_grid : {H}x{W} = {H * W} patches per image')
print(f'target_lyrs: {wrapper.target_layers}')

## SC 2 — Verify model properties

In [ ]:
assert wrapper.d_model > 0,                   'd_model must be positive'
assert wrapper.num_classes == 1000,            f'Expected 1000 classes, got {wrapper.num_classes}'
assert wrapper.config.patch_mode is True,      'patch_mode must be True for patch extraction'
assert wrapper.target_layers == TARGET_LAYERS, 'Target-layer mismatch'
print(f'SC 2 ✓  d_model={wrapper.d_model}, num_classes={wrapper.num_classes}, patch_mode=True')

## Load ImageNet val dataset and sample 10 random images

In [ ]:
transform   = wrapper.get_transform()
dataset     = datasets.ImageFolder(IMAGENET_VAL_DIR, transform=transform)
raw_dataset = datasets.ImageFolder(IMAGENET_VAL_DIR)  # no transform — PIL for visualization
class_names = dataset.classes

rng            = random.Random(SEED)
sample_indices = rng.sample(range(len(dataset)), N_IMAGES)

print(f'Val images : {len(dataset):,}  ({len(class_names)} classes)')
print(f'Sampled idx: {sample_indices}')

## SC 3 — Verify dataset loading and display sampled images

In [ ]:
img_chk, lbl_chk = dataset[sample_indices[0]]
assert img_chk.shape == (3, 224, 224), f'Expected (3,224,224), got {img_chk.shape}'
assert 0 <= lbl_chk < len(class_names), f'Label {lbl_chk} out of range'
print(f'Tensor shape: {tuple(img_chk.shape)}   label: {lbl_chk} ({class_names[lbl_chk]})')

fig, axes = plt.subplots(2, 5, figsize=(20, 9))
for ax, idx in zip(axes.flat, sample_indices):
    pil_img, lbl = raw_dataset[idx]
    ax.imshow(pil_img.resize((224, 224)))
    ax.set_title(f'{class_names[lbl][:18]}\nidx={idx}', fontsize=8)
    ax.axis('off')
plt.suptitle('10 Sampled ImageNet Val Images', fontsize=13)
plt.tight_layout()
plt.show()
print('SC 3 ✓')

## Extract patch tokens for all sampled images

In [ ]:
all_patch_states = {}  # {img_idx: {layer_idx: Tensor[1, H, W, d_model]}}
all_logits       = {}  # {img_idx: Tensor[1, num_classes]}
all_y_hats       = {}  # {img_idx: int}

for n, idx in enumerate(sample_indices):
    img_t, _ = dataset[idx]
    batch     = img_t.unsqueeze(0).to(DEVICE)        # [1, 3, 224, 224]

    patch_states, logits = wrapper.extract_patches(batch)

    all_patch_states[idx] = {k: v.cpu() for k, v in patch_states.items()}
    all_logits[idx]       = logits.cpu()
    all_y_hats[idx]       = int(logits[0].argmax())

    pred = class_names[all_y_hats[idx]]
    print(f'  [{n+1:2d}/{N_IMAGES}] idx={idx:6d}  y_hat={all_y_hats[idx]:4d}  ({pred})')

print('Extraction complete.')

## SC 4 — Verify patch tensor shapes and value ranges

In [ ]:
idx0 = sample_indices[0]
print(f'Checking shapes for image idx={idx0}:')
for layer_idx in TARGET_LAYERS:
    ps  = all_patch_states[idx0][layer_idx]
    exp = (1, H, W, wrapper.d_model)
    assert ps.shape == exp, f'Layer {layer_idx}: expected {exp}, got {ps.shape}'
    print(f'  Layer {layer_idx:2d}: shape={tuple(ps.shape)}  range=[{ps.min():.3f}, {ps.max():.3f}]  ✓')

lg = all_logits[idx0]
assert lg.shape == (1, wrapper.num_classes), f'Logits shape: {lg.shape}'
print(f'\nLogits shape  : {tuple(lg.shape)}')
print(f'y_hat for idx0: {all_y_hats[idx0]}  ({class_names[all_y_hats[idx0]]})')
print('SC 4 ✓')

## Load CLS lens checkpoints

In [ ]:
lenses = {}   # {layer_idx: BaseLens}
for layer_idx in TARGET_LAYERS:
    lenses[layer_idx] = load_lens_checkpoint(available_lenses[layer_idx], device=DEVICE)
    print(f'  Layer {layer_idx:2d}: {type(lenses[layer_idx]).__name__}')
print('CLS lenses loaded.')

## SC 5a — Verify checkpoint metadata (model name)

In [ ]:
for layer_idx in TARGET_LAYERS:
    payload  = torch.load(available_lenses[layer_idx], map_location='cpu', weights_only=False)
    meta     = payload.get('metadata', {})
    ckpt_mdl = meta.get('model_name', 'not stored')
    val_loss = meta.get('val_loss', float('nan'))
    epoch    = meta.get('epoch', '?')
    match    = 'OK' if ckpt_mdl in (MODEL_NAME, 'not stored') else f'MISMATCH (ckpt={ckpt_mdl})'
    print(f'  Layer {layer_idx:2d}: model={ckpt_mdl}  val_loss={val_loss:.4f}  epoch={epoch}  [{match}]')
    assert ckpt_mdl in (MODEL_NAME, 'not stored'), (
        f'Layer {layer_idx}: checkpoint trained on {ckpt_mdl} but model is {MODEL_NAME}'
    )
print('SC 5a ✓')

## SC 5 — Verify lens input/output dimensions

CLS lenses take a single token `[d_model]` as input (not a neighborhood).

In [ ]:
expected_in  = wrapper.d_model    # individual token, no neighborhood
expected_out = wrapper.num_classes
print(f'Expected lens input  = d_model    = {expected_in}')
print(f'Expected lens output = num_classes = {expected_out}\n')

for layer_idx, lens in lenses.items():
    if hasattr(lens, 'linear'):   # AffineLens
        w_in  = lens.linear.weight.shape[1]
        w_out = lens.linear.weight.shape[0]
    else:                          # MLPLens
        lins  = [m for m in lens.net.modules() if hasattr(m, 'weight') and m.weight.dim() == 2]
        w_in  = lins[0].weight.shape[1]
        w_out = lins[-1].weight.shape[0]

    in_ok  = 'OK' if w_in  == expected_in  else f'MISMATCH (got {w_in})'
    out_ok = 'OK' if w_out == expected_out else f'MISMATCH (got {w_out})'
    print(f'  Layer {layer_idx:2d}: in={w_in} [{in_ok}]   out={w_out} [{out_ok}]')
    assert w_in  == expected_in,  f'Layer {layer_idx} input dim mismatch'
    assert w_out == expected_out, f'Layer {layer_idx} output dim mismatch'

print('\nSC 5 ✓')

## Compute individual-patch score maps

For each patch `(i, j)`: feed `h[i,j]` directly to the CLS lens and record `softmax(logits)[y_hat]`.
All `H×W` patches are scored — no border exclusion, no NaN values.

In [ ]:
all_score_maps = {}  # {img_idx: {layer_idx: Tensor[H, W]}}

for img_idx in sample_indices:
    all_score_maps[img_idx] = {}
    y_hat = all_y_hats[img_idx]

    for layer_idx in TARGET_LAYERS:
        patches = all_patch_states[img_idx][layer_idx].to(DEVICE)   # [1, H, W, d]
        _, H_p, W_p, d = patches.shape

        flat = patches[0].reshape(H_p * W_p, d)     # [H*W, d_model]

        with torch.no_grad():
            out_logits = lenses[layer_idx](flat)     # [H*W, num_classes]
        scores = F.softmax(out_logits, dim=-1)[:, y_hat].cpu()   # [H*W]

        all_score_maps[img_idx][layer_idx] = scores.reshape(H_p, W_p)

print('Scoring complete for all images and layers.')

## SC 6 — Verify score map shapes, absence of NaN, and value ranges

In [ ]:
idx0 = sample_indices[0]
print(f'Checking score maps for image idx={idx0}:')
for layer_idx in TARGET_LAYERS:
    sm = all_score_maps[idx0][layer_idx]
    assert sm.shape == (H, W), f'Layer {layer_idx}: shape {sm.shape}'
    assert not torch.isnan(sm).any(), f'Layer {layer_idx}: unexpected NaN (all patches should be scored)'
    assert (sm >= 0.0).all() and (sm <= 1.0).all(), (
        f'Layer {layer_idx}: scores out of [0,1] — [{sm.min():.4f}, {sm.max():.4f}]'
    )
    print(
        f'  Layer {layer_idx:2d}  all {H*W} patches scored  '
        f'range=[{sm.min():.4f},{sm.max():.4f}]  mean={sm.mean():.4f}  ✓'
    )

print('\nSC 6 ✓')

## SC 7 — Quick sanity: verify a single patch manually

Pick patch (8, 8) in the first image at the first target layer and recompute its score by hand.

In [ ]:
idx0       = sample_indices[0]
layer0     = TARGET_LAYERS[0]
y_hat0     = all_y_hats[idx0]

# Manually score patch (8, 8)
pi, pj = 8, 8
single_tok = all_patch_states[idx0][layer0][0, pi, pj, :].unsqueeze(0).to(DEVICE)  # [1, d]
with torch.no_grad():
    manual_logits = lenses[layer0](single_tok)   # [1, C]
manual_score = float(F.softmax(manual_logits, dim=-1)[0, y_hat0])

stored_score = float(all_score_maps[idx0][layer0][pi, pj])

print(f'Manual score  for patch ({pi},{pj}) at layer {layer0}: {manual_score:.6f}')
print(f'Stored score  for patch ({pi},{pj}) at layer {layer0}: {stored_score:.6f}')
assert abs(manual_score - stored_score) < 1e-5, (
    f'Score mismatch: manual={manual_score:.6f}, stored={stored_score:.6f}'
)
print('SC 7 ✓  Manual and stored scores match')

## Visualize: score grid + heatmap overlay on original image

In [ ]:
for img_idx in sample_indices:
    y_hat       = all_y_hats[img_idx]
    pil_img, _  = raw_dataset[img_idx]
    img_np      = np.array(pil_img.resize((224, 224)))

    n_cols = len(TARGET_LAYERS) + 1
    fig, axes = plt.subplots(2, n_cols, figsize=(3.6 * n_cols, 8))

    # ── Column 0: original image ───────────────────────────────────────────
    axes[0, 0].imshow(img_np)
    axes[0, 0].set_title(f'Original  idx={img_idx}', fontsize=8)
    axes[0, 0].axis('off')
    axes[1, 0].text(
        0.5, 0.5, f'Pred:\n{class_names[y_hat][:22]}',
        ha='center', va='center', fontsize=8,
        transform=axes[1, 0].transAxes
    )
    axes[1, 0].axis('off')

    # ── Columns 1..N: per-layer score grid + overlay ───────────────────────
    for col, layer_idx in enumerate(TARGET_LAYERS, start=1):
        sm      = all_score_maps[img_idx][layer_idx].numpy()
        vmax_sm = float(sm.max()) if sm.max() > 1e-8 else 1.0

        # Top row: score grid at patch resolution
        im = axes[0, col].imshow(sm, cmap='hot', vmin=0.0, vmax=vmax_sm)
        axes[0, col].set_title(f'Layer {layer_idx}  Score Grid ({H}x{W})', fontsize=8)
        axes[0, col].axis('off')
        plt.colorbar(im, ax=axes[0, col], fraction=0.046, pad=0.04)

        # Bottom row: normalized heatmap overlay
        hm_norm = (sm - sm.min()) / (sm.max() - sm.min() + 1e-8)
        hm_u8   = (hm_norm * 255).astype(np.uint8)
        hm_224  = np.array(Image.fromarray(hm_u8).resize((224, 224), Image.NEAREST))
        axes[1, col].imshow(img_np)
        axes[1, col].imshow(hm_224, cmap='hot', alpha=0.55, vmin=0, vmax=255)
        axes[1, col].set_title(f'Layer {layer_idx}  Overlay', fontsize=8)
        axes[1, col].axis('off')

    title = f'CLS Lens (individual patches)  idx={img_idx}  Pred: {class_names[y_hat]}'
    plt.suptitle(title, fontsize=10)
    plt.tight_layout()
    plt.show()